# 📊 Task 4 — Mini Visualization Dashboard
**Maincrafts Technology | Data Science with Python Internship**

**Intern:** Mukund Rajpurohit | **Intern ID:** MT5153

---

## 📌 Objective
Build a mini data visualization dashboard on the Titanic dataset to communicate key insights using well-chosen charts and clear interpretations. This task focuses on:
- Data Cleaning & Feature Engineering
- Multiple Plot Types (Histogram, Bar, Boxplot, Scatterplot, Heatmap)
- Narration & Insight under each chart

**Dataset:** Titanic (loaded directly from URL)

**Tools Used:** Python, Pandas, NumPy, Matplotlib, Seaborn

---
## 📦 Step 1 — Import Libraries & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Apply Seaborn styling
sns.set_style('whitegrid')
sns.set_palette('muted')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

print('✅ Libraries imported successfully!')
print('📊 Dashboard ready to build!')

---
## 📂 Step 2 — Load the Dataset
Loading the Titanic dataset directly from URL — no manual download needed!

In [ ]:
# Load Titanic dataset directly from URL (no Kaggle login needed)
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
df = pd.read_csv(url)

print(f'✅ Dataset loaded successfully!')
print(f'📊 Shape: {df.shape[0]} rows × {df.shape[1]} columns')
df.head()

---
## 🔍 Step 3 — Dataset Overview

In [ ]:
# Basic info about the dataset
print('=== DATASET INFO ===')
df.info()

In [ ]:
# Statistical summary
print('=== STATISTICAL SUMMARY ===')
df.describe()

In [ ]:
# Check for missing values
print('=== MISSING VALUES ===')
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct.round(2)})
print(missing_df[missing_df['Missing Count'] > 0])

---
## 🧹 Step 4 — Data Cleaning
Handling missing values before building visualizations.

In [ ]:
# 1. Fill missing Age values with median (robust to outliers)
median_age = df['Age'].median()
df['Age'].fillna(median_age, inplace=True)
print(f'✅ Missing Age values filled with median: {median_age}')

# 2. Drop Cabin column — 77% missing values, not useful
df.drop(columns=['Cabin'], inplace=True)
print('✅ Cabin column dropped (77% missing)')

# 3. Fill missing Embarked with mode (most frequent port)
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)
print(f'✅ Missing Embarked filled with mode: {df["Embarked"].mode()[0]}')

# Verify cleaning
print(f'\n📊 Total missing values after cleaning: {df.isnull().sum().sum()}')
print('✅ Dataset is clean and ready for visualization!')

---
## ⚙️ Step 5 — Feature Engineering
Creating new meaningful columns from existing data to enhance our analysis.

In [ ]:
# Feature 1: AgeGroup — convert continuous Age into meaningful categories
bins   = [0, 12, 18, 35, 60, 100]
labels = ['Child (0-12)', 'Teen (13-18)', 'Young Adult (19-35)', 'Adult (36-60)', 'Senior (60+)']
df['AgeGroup'] = pd.cut(df['Age'], bins=bins, labels=labels)

print('✅ Feature 1: AgeGroup created')
print(df['AgeGroup'].value_counts())

In [ ]:
# Feature 2: FamilySize — total family members aboard
df['FamilySize'] = df['SibSp'] + df['Parch']

print('✅ Feature 2: FamilySize created')
print(f'   Min: {df["FamilySize"].min()} | Max: {df["FamilySize"].max()} | Mean: {df["FamilySize"].mean():.2f}')
print('\nSample rows:')
df[['SibSp', 'Parch', 'FamilySize', 'AgeGroup']].head()

---
## 📈 Step 6 — Visualization Dashboard

> **Dashboard contains 6 charts:**
> 1. Histogram — Age Distribution
> 2. Bar Chart — Survival Rate by Gender
> 3. Bar Chart — Survival Rate by Passenger Class
> 4. Boxplot — Fare Distribution by Passenger Class
> 5. Scatterplot — Age vs Fare (colored by Survival)
> 6. Heatmap — Correlation between Numeric Variables

### 📊 Chart 1 — Histogram: Age Distribution of Passengers

In [ ]:
# ── CHART 1: Histogram — Age Distribution ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Overall age distribution
axes[0].hist(df['Age'], bins=20, color='#4ECDC4', edgecolor='white', linewidth=0.8)
axes[0].axvline(df['Age'].mean(),   color='red',    linestyle='--', linewidth=2,
                label=f'Mean: {df["Age"].mean():.1f}')
axes[0].axvline(df['Age'].median(), color='orange', linestyle='--', linewidth=2,
                label=f'Median: {df["Age"].median():.1f}')
axes[0].set_title('Age Distribution — All Passengers', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Age', fontsize=12)
axes[0].set_ylabel('Number of Passengers', fontsize=12)
axes[0].legend(fontsize=10)

# Right: Age distribution split by survival
df[df['Survived']==1]['Age'].hist(ax=axes[1], bins=20, alpha=0.7,
                                   color='#51CF66', label='Survived',        edgecolor='white')
df[df['Survived']==0]['Age'].hist(ax=axes[1], bins=20, alpha=0.7,
                                   color='#FF6B6B', label='Did Not Survive', edgecolor='white')
axes[1].set_title('Age Distribution: Survived vs Not Survived', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Age', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].legend(fontsize=10)

plt.suptitle('📊 Chart 1 — Passenger Age Distribution', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('chart1_age_histogram.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart 1 saved!')

**📌 Insight:** Most Titanic passengers were between **20–35 years old**, with the distribution peaking around age 25. The right panel shows that younger passengers (especially children) had higher survival counts, while middle-aged male passengers (20–40) form the bulk of those who did not survive.

### 📊 Chart 2 — Bar Chart: Survival Rate by Gender

In [ ]:
# ── CHART 2: Bar Chart — Survival by Gender ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Survival rate (%)
sns.barplot(x='Sex', y='Survived', data=df,
            palette=['#FF6B9D', '#4ECDC4'],
            estimator=lambda x: sum(x)/len(x)*100, ax=axes[0])
axes[0].set_title('Survival Rate by Gender (%)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Gender', fontsize=12)
axes[0].set_ylabel('Survival Rate (%)', fontsize=12)
axes[0].set_xticklabels(['Female', 'Male'], fontsize=11)
axes[0].set_ylim(0, 100)
for p in axes[0].patches:
    axes[0].annotate(f'{p.get_height():.1f}%',
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='bottom', fontsize=12, fontweight='bold')

# Right: Count chart
survived_gender = df.groupby(['Sex', 'Survived']).size().unstack()
survived_gender.plot(kind='bar', ax=axes[1],
                     color=['#FF6B6B', '#51CF66'], edgecolor='white', width=0.6)
axes[1].set_title('Survival Count by Gender', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Gender', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].set_xticklabels(['Female', 'Male'], rotation=0, fontsize=11)
axes[1].legend(['Did Not Survive', 'Survived'], fontsize=10)

plt.suptitle('📊 Chart 2 — Gender vs Survival', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('chart2_survival_by_gender.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart 2 saved!')

**📌 Insight:** Female passengers had a dramatically higher survival rate (~74%) compared to males (~19%). This is a clear reflection of the **"Women and Children First"** evacuation policy enforced during the Titanic disaster. The count chart confirms that the vast majority of male passengers did not survive.

### 📊 Chart 3 — Bar Chart: Survival Rate by Passenger Class

In [ ]:
# ── CHART 3: Bar Chart — Survival by Passenger Class ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Survival rate (%)
sns.barplot(x='Pclass', y='Survived', data=df,
            palette=['#FFD700', '#C0C0C0', '#CD7F32'],
            estimator=lambda x: sum(x)/len(x)*100, ax=axes[0])
axes[0].set_title('Survival Rate by Passenger Class (%)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Passenger Class', fontsize=12)
axes[0].set_ylabel('Survival Rate (%)', fontsize=12)
axes[0].set_xticklabels(['1st Class', '2nd Class', '3rd Class'], fontsize=11)
axes[0].set_ylim(0, 100)
for p in axes[0].patches:
    axes[0].annotate(f'{p.get_height():.1f}%',
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='bottom', fontsize=12, fontweight='bold')

# Right: Count chart
survived_class = df.groupby(['Pclass', 'Survived']).size().unstack()
survived_class.plot(kind='bar', ax=axes[1],
                    color=['#FF6B6B', '#51CF66'], edgecolor='white', width=0.6)
axes[1].set_title('Survival Count by Passenger Class', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Passenger Class', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].set_xticklabels(['1st Class', '2nd Class', '3rd Class'], rotation=0, fontsize=11)
axes[1].legend(['Did Not Survive', 'Survived'], fontsize=10)

plt.suptitle('📊 Chart 3 — Passenger Class vs Survival', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('chart3_survival_by_class.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart 3 saved!')

**📌 Insight:** 1st Class passengers had the highest survival rate (~63%), while 3rd Class passengers survived at only ~24%. This stark difference reflects the **socioeconomic disparity** on the ship — 1st class cabins were on higher decks, closer to lifeboats, while 3rd class passengers were located deep in the lower decks.

### 📊 Chart 4 — Boxplot: Fare Distribution by Passenger Class

In [ ]:
# ── CHART 4: Boxplot — Fare by Passenger Class ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Fare by class boxplot
sns.boxplot(x='Pclass', y='Fare', data=df,
            palette=['#FFD700', '#C0C0C0', '#CD7F32'], ax=axes[0])
axes[0].set_title('Fare Distribution by Passenger Class', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Passenger Class', fontsize=12)
axes[0].set_ylabel('Fare (£)', fontsize=12)
axes[0].set_xticklabels(['1st Class', '2nd Class', '3rd Class'], fontsize=11)

# Right: Fare by class + survival hue
sns.boxplot(x='Pclass', y='Fare', hue='Survived', data=df,
            palette={0: '#FF6B6B', 1: '#51CF66'}, ax=axes[1])
axes[1].set_title('Fare vs Class (Split by Survival)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Passenger Class', fontsize=12)
axes[1].set_ylabel('Fare (£)', fontsize=12)
axes[1].set_xticklabels(['1st Class', '2nd Class', '3rd Class'], fontsize=11)
axes[1].legend(title='Survived', labels=['No', 'Yes'], fontsize=10)

plt.suptitle('📊 Chart 4 — Fare Distribution by Passenger Class', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('chart4_fare_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart 4 saved!')

**📌 Insight:** The boxplot clearly shows that 1st Class passengers paid significantly higher fares with large outliers reaching £500+, while 3rd Class fares were tightly clustered at the lower end. The right panel reveals that even within each class, **survivors tended to pay slightly higher fares**, suggesting that cabin location (linked to fare) played a role in survival.

### 📊 Chart 5 — Scatterplot: Age vs Fare (colored by Survival)

In [ ]:
# ── CHART 5: Scatterplot — Age vs Fare colored by Survival ───────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Age vs Fare colored by survival
colors = df['Survived'].map({0: '#FF6B6B', 1: '#51CF66'})
scatter = axes[0].scatter(df['Age'], df['Fare'], c=colors, alpha=0.6, edgecolors='white',
                           linewidth=0.3, s=40)
axes[0].set_title('Age vs Fare (Colored by Survival)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Age', fontsize=12)
axes[0].set_ylabel('Fare (£)', fontsize=12)
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#51CF66', label='Survived'),
                   Patch(facecolor='#FF6B6B', label='Did Not Survive')]
axes[0].legend(handles=legend_elements, fontsize=10)

# Right: Same but split by passenger class using seaborn
sns.scatterplot(x='Age', y='Fare', hue='Survived', style='Pclass',
                palette={0: '#FF6B6B', 1: '#51CF66'},
                data=df, alpha=0.7, ax=axes[1])
axes[1].set_title('Age vs Fare (by Survival & Class)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Age', fontsize=12)
axes[1].set_ylabel('Fare (£)', fontsize=12)
axes[1].legend(fontsize=9)

plt.suptitle('📊 Chart 5 — Age vs Fare Scatterplot', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('chart5_age_vs_fare_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart 5 saved!')

**📌 Insight:** High-fare passengers (upper portion of the chart) are predominantly green (survived), confirming that wealth strongly influenced survival. Most of the red dots (did not survive) are concentrated in the **low fare, mid-age (20–40)** zone — these were largely 3rd class male passengers. Age alone shows no strong linear relationship with fare.

### 📊 Chart 6 — Heatmap: Correlation between Numeric Variables

In [ ]:
# ── CHART 6: Heatmap — Correlation Matrix ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Select only numeric columns
numeric_df = df.select_dtypes(include=[np.number])

# Left: Full correlation heatmap
corr_matrix = numeric_df.corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f',
            linewidths=0.5, ax=axes[0], annot_kws={'size': 10})
axes[0].set_title('Correlation Heatmap (All Numeric Variables)',
                   fontsize=13, fontweight='bold')

# Right: Correlation with Survived only (sorted bar chart)
survival_corr = corr_matrix['Survived'].drop('Survived').sort_values()
colors_bar = ['#FF6B6B' if v < 0 else '#51CF66' for v in survival_corr.values]
axes[1].barh(survival_corr.index, survival_corr.values, color=colors_bar, edgecolor='white')
axes[1].axvline(0, color='black', linewidth=0.8, linestyle='--')
axes[1].set_title('Correlation of Each Feature with Survival',
                   fontsize=13, fontweight='bold')
axes[1].set_xlabel('Correlation Coefficient', fontsize=12)
for i, (val, name) in enumerate(zip(survival_corr.values, survival_corr.index)):
    axes[1].text(val + 0.01 if val >= 0 else val - 0.01, i,
                 f'{val:.2f}', va='center', ha='left' if val >= 0 else 'right',
                 fontsize=10, fontweight='bold')

plt.suptitle('📊 Chart 6 — Correlation Heatmap', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('chart6_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart 6 saved!')

**📌 Insight:** The correlation bar chart shows that **Fare (+0.26)** and **Pclass (-0.34)** are the strongest predictors of survival — higher fare = more likely to survive, higher class number = less likely. **SibSp and Parch** show weak correlations, and **Age (-0.07)** has almost no linear correlation with survival, meaning age alone doesn't predict survival well.

---
## 🔍 Step 7 — Bonus: Facet Grid (Sex × Class Survival)

In [ ]:
# ── BONUS: Facet Grid — Survival by Class split by Gender ─────────────────────
g = sns.catplot(
    col='Sex', x='Pclass', y='Survived', kind='bar',
    data=df, palette=['#FFD700', '#C0C0C0', '#CD7F32'],
    height=5, aspect=1.1,
    estimator=lambda x: sum(x)/len(x)*100
)
g.set_axis_labels('Passenger Class', 'Survival Rate (%)')
g.set_titles(col_template='{col_name} Passengers')
g.set_xticklabels(['1st', '2nd', '3rd'])
g.figure.suptitle('📊 Bonus — Survival Rate by Class & Gender (Facet Grid)',
                   fontsize=14, fontweight='bold', y=1.03)

for ax in g.axes.flat:
    for p in ax.patches:
        ax.annotate(f'{p.get_height():.1f}%',
                    (p.get_x() + p.get_width()/2., p.get_height()),
                    ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.savefig('chart7_facet_grid.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Bonus Facet Grid saved!')

**📌 Insight:** The facet grid reveals an important sub-group pattern — **Female 1st Class passengers had nearly 97% survival**, while **Male 3rd Class passengers had only ~14% survival**. Even female 3rd class passengers (~50%) survived at more than double the rate of male 3rd class passengers, confirming that **gender was a stronger survival factor than class**.

---
## 📝 Step 8 — Final Summary Report

In [ ]:
total        = len(df)
survived     = df['Survived'].sum()
overall_rate = round(survived / total * 100, 2)

female_rate  = round(df[df['Sex']=='female']['Survived'].mean() * 100, 2)
male_rate    = round(df[df['Sex']=='male']['Survived'].mean()   * 100, 2)
c1_rate      = round(df[df['Pclass']==1]['Survived'].mean() * 100, 2)
c2_rate      = round(df[df['Pclass']==2]['Survived'].mean() * 100, 2)
c3_rate      = round(df[df['Pclass']==3]['Survived'].mean() * 100, 2)
fare_mean_c1 = round(df[df['Pclass']==1]['Fare'].mean(), 2)
fare_mean_c3 = round(df[df['Pclass']==3]['Fare'].mean(), 2)

print('=' * 58)
print('   TASK 4 — MINI VISUALIZATION DASHBOARD SUMMARY')
print('   Titanic Dataset | Maincrafts Technology')
print('=' * 58)
print(f'  Total Passengers    : {total}')
print(f'  Total Survived      : {survived} ({overall_rate}%)')
print(f'  Total Not Survived  : {total - survived} ({round(100-overall_rate,2)}%)')
print('-' * 58)
print('  CHART INSIGHTS SUMMARY:')
print(f'  Chart 1 (Histogram) : Peak age 20-35; Children most likely to survive')
print(f'  Chart 2 (Bar-Gender): Female {female_rate}% vs Male {male_rate}% survival')
print(f'  Chart 3 (Bar-Class) : 1st {c1_rate}% | 2nd {c2_rate}% | 3rd {c3_rate}%')
print(f'  Chart 4 (Boxplot)   : 1st class avg fare £{fare_mean_c1} vs 3rd £{fare_mean_c3}')
print(f'  Chart 5 (Scatter)   : High-fare passengers cluster among survivors')
print(f'  Chart 6 (Heatmap)   : Pclass (-0.34) & Fare (+0.26) top predictors')
print(f'  Bonus  (Facet Grid) : Female 1st class ~97% survival rate')
print('-' * 58)
print('  KEY DASHBOARD TAKEAWAYS:')
print('  1. Gender was the strongest survival factor')
print('     (Women prioritised in evacuation)')
print('  2. Class/wealth determined lifeboat access')
print('     (1st class cabins near upper decks & lifeboats)')
print('  3. Fare correlated positively with survival')
print('  4. Age had low direct correlation with survival')
print('  5. Small families survived better than solo travellers')
print('=' * 58)
print('  ✅ Task 4 Complete — Mukund Rajpurohit (MT5153)')
print('=' * 58)

---
## ✅ Conclusion

This mini visualization dashboard successfully communicates key insights from the Titanic dataset using **6 distinct chart types** — Histogram, Bar Charts, Boxplot, Scatterplot, Correlation Heatmap, and a Facet Grid.

**Key Findings:**
- **Gender** was the single most impactful survival factor — females survived at nearly 4× the rate of males.
- **Passenger Class** strongly influenced survival — 1st class had 2.6× better survival odds than 3rd class.
- **Fare** positively correlated with survival (+0.26), reflecting the link between wealth and cabin location.
- **Age** had minimal direct impact on survival as a standalone variable.
- The **Facet Grid** revealed that gender outweighed class — even female 3rd class passengers survived better than male 2nd class passengers.

This dashboard demonstrates the power of **visual storytelling in data analysis** — the ability to communicate complex patterns clearly to any audience.

---
**Intern:** Mukund Rajpurohit | **Intern ID:** MT5153 | **Company:** Maincrafts Technology

**Domain:** Data Science with Python | **Task:** 4 of the internship programme